In [1]:
import shutil
import subprocess
import sys

print("Python:", sys.version.split()[0])
print("Disk free GiB:", round(shutil.disk_usage("/content").free / 2**30, 1))
print(
    subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True
    ).stdout
)
print(subprocess.run(["free", "-h"], capture_output=True, text=True).stdout)
print(
    "GitHub:",
    subprocess.run(
        ["git", "ls-remote", "https://github.com/PohjoinenJohtaja/llm-project.git", "HEAD"],
        capture_output=True,
        text=True,
        timeout=30,
    ).stdout,
)

Python: 3.13.16
Disk free GiB: 112.5
Tesla T4, 15360 MiB

               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.1Gi       8.8Gi       2.4Mi       3.1Gi        11Gi
Swap:             0B          0B          0B

GitHub: 00002631b4c8f2b136d2103ce43fa1ed6a0363db	HEAD



In [2]:
import os
import pathlib
import subprocess


def run(cmd, cwd=None):
    print("RUN:", " ".join(cmd), flush=True)
    subprocess.run(cmd, cwd=cwd, check=True)


run(["pip", "-q", "install", "uv"])
repo = pathlib.Path("/content/llm-project")
if not repo.exists():
    run(["git", "clone", "https://github.com/PohjoinenJohtaja/llm-project.git", str(repo)])
run(["uv", "sync", "--python", "3.11"], cwd=repo)
print((repo / ".env.example").read_text())
run(
    [
        "bash",
        "-c",
        "curl -fsSL https://ollama.com/install.sh -o /tmp/install-ollama.sh "
        "&& OLLAMA_VERSION=0.15.4 sh /tmp/install-ollama.sh",
    ]
)
print("SETUP_COMPLETE")

RUN: pip -q install uv
RUN: git clone https://github.com/PohjoinenJohtaja/llm-project.git /content/llm-project
RUN: uv sync --python 3.11
# Скопируйте файл в .env. Файл .env в репозиторий не попадает.

# Адрес модельного сервера с OpenAI-совместимым API.
# Ollama запущена на вашем компьютере (вариант для macOS и Windows):
LLM_BASE_URL=http://host.docker.internal:11434/v1
# Ollama запущена в контейнере командой make up-llm:
# LLM_BASE_URL=http://llm:11434/v1

LLM_MODEL=qwen3:4b-instruct-2507-q4_K_M
EMBED_MODEL=bge-m3
LLM_API_KEY=ollama

# Доля запросов, на которые ERP отвечает ошибкой 503. Для проверки повторов поставьте 0.3.
ERP_FAIL_RATE=0

RUN: bash -c curl -fsSL https://ollama.com/install.sh -o /tmp/install-ollama.sh && OLLAMA_VERSION=0.15.4 sh /tmp/install-ollama.sh


CalledProcessError: Command '['bash', '-c', 'curl -fsSL https://ollama.com/install.sh -o /tmp/install-ollama.sh && OLLAMA_VERSION=0.15.4 sh /tmp/install-ollama.sh']' returned non-zero exit status 1.

In [3]:
r = subprocess.run(["bash", "-c", "OLLAMA_VERSION=0.15.4 sh /tmp/install-ollama.sh"], capture_output=True, text=True)
print(r.stdout[-4000:])
print(r.stderr[-4000:])
print("EXIT", r.returncode)

ERROR: This version requires zstd for extraction. Please install zstd and try again:
  - Debian/Ubuntu: sudo apt-get install zstd
  - RHEL/CentOS/Fedora: sudo dnf install zstd
  - Arch: sudo pacman -S zstd

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local

EXIT 1


In [4]:
def logged(cmd, cwd=None):
    r = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(r.stdout[-5000:], flush=True)
    r.check_returncode()


logged(["bash", "-c", "apt-get update -qq && apt-get install -y -qq zstd"])
logged(["bash", "-c", "OLLAMA_VERSION=0.15.4 sh /tmp/install-ollama.sh"])
print("OLLAMA_INSTALLED")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 
(Reading database ... 5%
(Reading database ... 10%
(Reading database ... 15%
(Reading database ... 20%
(Reading database ... 25%
(Reading database ... 30%
(Reading database ... 35%
(Reading database ... 40%
(Reading database ... 45%
(Reading database ... 50%
(Reading database ... 55%
(Reading database ... 60%
(Reading database ... 65%
(Reading database ... 70%
(Reading database ... 75%
(Reading database ... 80%
(Reading database ... 85%
(Reading database ... 90%
(Reading database ... 95%
(Reading database ... 100%
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) .

In [5]:
import json
import time

import requests

art = repo / "artifacts"
art.mkdir(exist_ok=True)
(repo / ".env").write_text(
    (repo / ".env.example").read_text().replace("http://host.docker.internal:11434/v1", "http://localhost:11434/v1")
)
env = os.environ.copy()
env["OLLAMA_HOST"] = "127.0.0.1:11434"
env["OLLAMA_NUM_PARALLEL"] = "1"
ollama_log = open(art / "colab-ollama.log", "a")
ollama_proc = subprocess.Popen(["ollama", "serve"], env=env, stdout=ollama_log, stderr=subprocess.STDOUT)
for _i in range(60):
    try:
        response = requests.get("http://localhost:11434/api/version", timeout=2)
        response.raise_for_status()
        print(response.json())
        break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start")
for model in ["qwen3:4b-instruct-2507-q4_K_M", "bge-m3"]:
    print("Downloading", model, flush=True)
    with requests.post(
        "http://localhost:11434/api/pull", json={"name": model, "stream": True}, stream=True, timeout=600
    ) as response:
        response.raise_for_status()
        for line in response.iter_lines():
            if line:
                status = json.loads(line)
                if "error" in status:
                    raise RuntimeError(status["error"])
        print(model, status, flush=True)
erp_log = open(art / "colab-erp.log", "a")
erp_proc = subprocess.Popen(
    ["uv", "run", "uvicorn", "erp.app:app", "--host", "127.0.0.1", "--port", "8100"],
    cwd=repo,
    stdout=erp_log,
    stderr=subprocess.STDOUT,
)
logged(["uv", "run", "python", "--version"], cwd=repo)
print("MODELS_READY")

{'version': '0.15.4'}
qwen3:4b-instruct-2507-q4_K_M {'status': 'success'}
bge-m3 {'status': 'success'}
Python 3.11.18

MODELS_READY


In [6]:
t = time.time()
r = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "qwen3:4b-instruct-2507-q4_K_M",
        "prompt": "Reply OK",
        "stream": False,
        "options": {"num_predict": 8},
    },
    timeout=120,
)
r.raise_for_status()
print("MODEL_REPLY:", r.json().get("response"), "SECONDS:", round(time.time() - t, 1))
logged(["ollama", "ps"])
print("ERP:", requests.get("http://127.0.0.1:8100/openapi.json", timeout=10).status_code)
print((repo / "Makefile").read_text())

MODEL_REPLY: OK SECONDS: 90.2
NAME                             ID              SIZE      PROCESSOR    CONTEXT    UNTIL              
qwen3:4b-instruct-2507-q4_K_M    0edcdef34593    3.6 GB    100% GPU     4096       4 minutes from now    

ERP: 200
# Команды проекта. Файл остаётся как есть.
# По умолчанию всё выполняется в контейнере. Чтобы работать без Docker, передайте RUN="uv run":
#   make test RUN="uv run"
RUN ?= docker compose run --rm app
MODE ?= agent
ARGS ?=
Q ?= Ошибка E-17 на AR-210. Какую деталь заказать, есть ли она на складе в Казани и сколько она стоит?

include .env
export

.PHONY: up up-llm pull-models index ask eval eval-all eval-extraction test lint down

.env:
	cp .env.example .env

up: .env ## собрать и запустить сервисы
	docker compose up -d --build app erp

up-llm: .env ## запустить модельный сервер в контейнере
	docker compose --profile local-llm up -d llm

pull-models: ## скачать модели в контейнер llm
	docker compose --profile local-llm exec llm ollama pull $(

In [7]:
# Baseline: journal and final report are preserved in artifacts.
from datetime import UTC, datetime

assert "baseline_proc" not in globals() or globals()["baseline_proc"].poll() is not None, "Baseline already running"
log_path = art / ("colab-baseline-" + datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ") + ".log")
baseline_log = open(log_path, "w")
baseline_proc = subprocess.Popen(
    ["make", "eval", "MODE=baseline", "RUN=uv run"],
    cwd=repo,
    stdout=baseline_log,
    stderr=subprocess.STDOUT,
    env={**os.environ, "PYTHONUNBUFFERED": "1"},
)
started = time.time()
print("BASELINE_STARTED", "PID", baseline_proc.pid, "LOG", str(log_path), flush=True)
while baseline_proc.poll() is None:
    time.sleep(30)
    print("Elapsed seconds:", int(time.time() - started), "Process:", baseline_proc.poll(), flush=True)
baseline_log.close()
print("BASELINE_EXIT", baseline_proc.returncode)
print(log_path.read_text()[-8000:])
report = art / "eval_baseline.json"
if report.exists():
    print("BASELINE_REPORT_JSON")
    print(report.read_text())
    print("ARCHIVE", shutil.make_archive("/content/llm-project-baseline-artifacts", "zip", repo, "artifacts"))
else:
    print("No final report generated")

BASELINE_STARTED PID 3547 LOG /content/llm-project/artifacts/colab-baseline-20261124T142447Z.log
Elapsed seconds: 30 Process: None
Elapsed seconds: 60 Process: 2
BASELINE_EXIT 2
uv run python -m assistant eval --mode baseline
Ошибка: сервис модели ответил кодом 500: {"error":{"message":"failed to encode response: json: unsupported value: NaN","type":"api_error","param":null,"code":null}}

make: *** [Makefile:34: eval] Error 1

No final report generated


In [8]:
print((art / "colab-ollama.log").read_text()[-11000:])
print("CACHE FILES", len(list((art / "llm_cache").glob("*"))))
for p in (repo / "src").rglob("*.py"):
    if "def embed" in p.read_text():
        print("EMBED SOURCE", p, p.read_text()[:14000])

NaN"
init: embeddings required but some input tokens were not marked as outputs -> overriding
time=2026-11-24T14:25:37.880Z level=INFO source=server.go:1748 msg="llm embedding error: failed to encode response: json: unsupported value: NaN"
init: embeddings required but some input tokens were not marked as outputs -> overriding
time=2026-11-24T14:25:37.901Z level=INFO source=server.go:1748 msg="llm embedding error: failed to encode response: json: unsupported value: NaN"
init: embeddings required but some input tokens were not marked as outputs -> overriding
time=2026-11-24T14:25:37.923Z level=INFO source=server.go:1748 msg="llm embedding error: failed to encode response: json: unsupported value: NaN"
init: embeddings required but some input tokens were not marked as outputs -> overriding
time=2026-11-24T14:25:37.936Z level=INFO source=server.go:1748 msg="llm embedding error: failed to encode response: json: unsupported value: NaN"
init: embeddings required but some input tokens were no

In [9]:
import math

probe_results = []
for label, options in [("GPU default", {}), ("CPU", {"num_gpu": 0})]:
    t = time.time()
    r = requests.post(
        "http://localhost:11434/api/embed",
        json={"model": "bge-m3", "input": ["Проверка двигателя AR-210."], "options": options, "keep_alive": 0},
        timeout=180,
    )
    result = {"mode": label, "status": r.status_code, "seconds": round(time.time() - t, 1)}
    if r.ok:
        v = r.json()["embeddings"][0]
        result.update(dim=len(v), finite=all(math.isfinite(x) for x in v), norm=sum(x * x for x in v) ** 0.5)
    else:
        result["error"] = r.text[:300]
    probe_results.append(result)
    print(result, flush=True)
(art / "embedding-diagnostic.json").write_text(json.dumps(probe_results, indent=2))

{'mode': 'GPU default', 'status': 200, 'seconds': 15.4, 'dim': 1024, 'finite': True, 'norm': 1.0000000417774462}
{'mode': 'CPU', 'status': 200, 'seconds': 5.0, 'dim': 1024, 'finite': True, 'norm': 1.000000182452942}


280

In [10]:
for p in (repo / "src/assistant").rglob("*.py"):
    s = p.read_text()
    if "class Baseline" in s or "def load_chunks" in s:
        print(p, s[:10000])
print("CLI FILES", [str(p.relative_to(repo)) for p in (repo / "src/assistant").glob("*.py")])

/content/llm-project/src/assistant/infra/chunking.py """Разбиение документов корпуса на чанки."""

from pathlib import Path

from assistant.infra.schemas import Chunk

MAX_CHARS = 1000


def chunk_document(doc_id: str, text: str, max_chars: int = MAX_CHARS) -> list[Chunk]:
    """Делит документ по заголовкам второго уровня и склеивает соседние разделы, пока они помещаются.

    Каждый чанк начинается с заголовка документа: без него из раздела «Состав узлов» не понять,
    о какой модели речь.
    """
    lines = text.strip().splitlines()
    title = lines[0] if lines and lines[0].startswith("# ") else ""
    sections: list[str] = []
    current: list[str] = []
    for line in lines[1:] if title else lines:
        if line.startswith("## ") and current:
            sections.append("\n".join(current).strip())
            current = []
        current.append(line)
    if current:
        sections.append("\n".join(current).strip())
    packed: list[str] = []
    for section in sections:
   

In [11]:
capture_code = """import json,sys,runpy
from pathlib import Path
from assistant.infra.llm import LLMClient
def capture(self,path,body):
    Path('artifacts/diagnostic-request.json').write_text(json.dumps({'path':path,'body':body},ensure_ascii=False))
    print('CAPTURED',path,'items',len(body.get('input',[])))
    raise SystemExit(0)
LLMClient._post=capture
sys.argv=['assistant','eval','--mode','baseline']
runpy.run_module('assistant',run_name='__main__')
"""
logged(["uv", "run", "python", "-c", capture_code], cwd=repo)
original = json.loads((art / "diagnostic-request.json").read_text())
texts = original["body"]["input"]
print("DOCUMENTS", len(texts), "MIN/MAX CHARS", min(map(len, texts)), max(map(len, texts)))
for label, options in [("GPU corpus", {}), ("CPU corpus", {"num_gpu": 0})]:
    t = time.time()
    r = requests.post(
        "http://localhost:11434/api/embed",
        json={"model": "bge-m3", "input": texts, "options": options, "keep_alive": 0},
        timeout=300,
    )
    result = {"mode": label, "status": r.status_code, "seconds": round(time.time() - t, 1)}
    if r.ok:
        vs = r.json()["embeddings"]
        result.update(vectors=len(vs), finite=all(math.isfinite(x) for v in vs for x in v))
    else:
        result["error"] = r.text[:300]
    probe_results.append(result)
    print(result, flush=True)
    (art / "embedding-diagnostic.json").write_text(json.dumps(probe_results, indent=2))

CAPTURED embeddings items 92

DOCUMENTS 92 MIN/MAX CHARS 300 990
{'mode': 'GPU corpus', 'status': 500, 'seconds': 5.1, 'error': '{"error":"failed to encode response: json: unsupported value: NaN"}'}
{'mode': 'CPU corpus', 'status': 200, 'seconds': 273.6, 'vectors': 92, 'finite': True}


In [12]:
import hashlib

assert baseline_proc.poll() is not None, "Baseline already running"
modelfile = art / "Modelfile.bge-m3-cpu"
modelfile.write_text("FROM bge-m3\nPARAMETER num_gpu 0\n")
logged(["ollama", "create", "bge-m3-cpu", "-f", str(modelfile)])
(repo / ".env").write_text((repo / ".env").read_text().replace("EMBED_MODEL=bge-m3", "EMBED_MODEL=bge-m3-cpu"))
# Reuse the 92 verified CPU vectors from the diagnostic, with matching cache keys.
assert probe_results[-1]["mode"] == "CPU corpus" and probe_results[-1]["finite"]
assert len(texts) == len(vs) == 92 and all(len(v) == 1024 and all(math.isfinite(x) for x in v) for v in vs)
cache = art / "llm_cache"
cache.mkdir(exist_ok=True)
for text, vector in zip(texts, vs, strict=True):
    raw = json.dumps(
        {"path": "embeddings", "body": {"model": "bge-m3-cpu", "input": text}}, ensure_ascii=False, sort_keys=True
    )
    key = hashlib.sha256(raw.encode()).hexdigest()
    (cache / (key + ".json")).write_text(json.dumps({"embedding": vector}, allow_nan=False))
print("Cached verified CPU vectors:", len(vs))
r = requests.post(
    "http://localhost:11434/v1/embeddings",
    json={"model": "bge-m3-cpu", "input": ["Проверка двигателя AR-210."]},
    timeout=120,
)
r.raise_for_status()
vector = r.json()["data"][0]["embedding"]
assert len(vector) == 1024 and all(math.isfinite(x) for x in vector)
print("OpenAI embeddings API: OK")
logged(["ollama", "ps"])
(art / "cpu-workaround.json").write_text(
    json.dumps(
        {
            "base_model": "bge-m3",
            "alias": "bge-m3-cpu",
            "num_gpu": 0,
            "seeded_cpu_vectors": 92,
            "generation_model": "qwen3:4b-instruct-2507-q4_K_M",
        },
        indent=2,
    )
)

gathering model components gathering model components gathering model components 
using existing layer sha256:daec91ffb5dd0c27411bd71f29932917c49cf529a641d0168496c3a501e3062c 
using existing layer sha256:a406579cd136771c705c521db86ca7d60a6f3de7c9b5460e6193a2df27861bde 
creating new layer sha256:70701d1fff6d0ee0d7c41bb24fcc0021e5ad086027fe9dceefb89347d2d2f7bf 
writing manifest 
success 

Cached verified CPU vectors: 92
OpenAI embeddings API: OK
NAME                 ID              SIZE      PROCESSOR    CONTEXT    UNTIL              
bge-m3-cpu:latest    b5ebf672121b    1.1 GB    100% CPU     4096       4 minutes from now    



152

In [13]:
assert baseline_proc.poll() is not None, "Baseline already running"
log_path = art / ("colab-baseline-cpu-embed-" + datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ") + ".log")
baseline_log = open(log_path, "w")
baseline_proc = subprocess.Popen(
    ["make", "eval", "MODE=baseline", "RUN=uv run"],
    cwd=repo,
    stdout=baseline_log,
    stderr=subprocess.STDOUT,
    env={**os.environ, "PYTHONUNBUFFERED": "1"},
)
started = time.time()
print("BASELINE_STARTED", baseline_proc.pid, "LOG", str(log_path), flush=True)
while baseline_proc.poll() is None:
    time.sleep(30)
    answers = 0
    for p in cache.glob("*.json"):
        try:
            answers += int("choices" in json.loads(p.read_text()))
        except (ValueError, OSError):
            pass
    print(
        "Elapsed:",
        int(time.time() - started),
        "sec; cached answers:",
        answers,
        "/24; exit:",
        baseline_proc.poll(),
        flush=True,
    )
    if int(time.time() - started) < 60:
        print(subprocess.run(["ollama", "ps"], capture_output=True, text=True).stdout, flush=True)
baseline_log.close()
print("BASELINE_EXIT", baseline_proc.returncode)
print(log_path.read_text()[-8000:])
report = art / "eval_baseline.json"
if report.exists():
    print("BASELINE_REPORT_JSON")
    print(report.read_text())
    print("ARCHIVE", shutil.make_archive("/content/llm-project-baseline-artifacts", "zip", repo, "artifacts"))

BASELINE_STARTED 6999 LOG /content/llm-project/artifacts/colab-baseline-cpu-embed-20261124T143741Z.log
Elapsed: 30 sec; cached answers: 8 /24; exit: None
NAME                             ID              SIZE      PROCESSOR    CONTEXT    UNTIL              
bge-m3-cpu:latest                b5ebf672121b    1.1 GB    100% CPU     4096       4 minutes from now    
qwen3:4b-instruct-2507-q4_K_M    0edcdef34593    3.6 GB    100% GPU     4096       4 minutes from now    

Elapsed: 60 sec; cached answers: 23 /24; exit: None
Elapsed: 90 sec; cached answers: 24 /24; exit: 0
BASELINE_EXIT 0
uv run python -m assistant eval --mode baseline
baseline: верных ответов 0.50 (local 1.00, multihop 0.57, global 0.00, tool 0.00, refusal 1.00), выдуманные факты 0.00, вызовов модели на вопрос 1.0

BASELINE_REPORT_JSON
{
  "summary": {
    "mode": "baseline",
    "model": "qwen3:4b-instruct-2507-q4_K_M",
    "questions": 24,
    "accuracy": 0.5,
    "by_type": {
      "local": {
        "questions": 5,
       